## Open Government Data, provided by **OpenDataZurich**
*Autogenerated Python starter code for data set with identifier* **geo_zueri_wie_neu**

## Dataset
# **Züri wie neu**

## Description

Züri wie neu Meldungen: Gemeldet werden können sämtliche Schäden an der Infrastruktur der Stadt Zürich. Dabei kann es sich um ein Loch im Strassenbelag, ein Graffiti am Stadthaus oder eine durch Vandalen beschädigte Parkbank handeln. 

**Zweck**: «Züri wie neu» ist eine Online-Plattform, über die die Einwohnerinnen und Einwohner der Stadt Zürich auf Schäden an der städtischen Infrastruktur hinweisen können. «Züri wie neu» wird von der Stadtverwaltung moderiert und transparent geführt. Die Meldungen werden innerhalb von einem Arbeitstag den zuständigen Fachstellen zugewiesen und innert fünf Arbeitstagen abschliessend beantwortet. Fällt eine Meldung nicht in den Zuständigkeitsbereich der Stadtverwaltung, wird die Meldung anonymisiert der zuständigen dritten Stelle per E-Mail zugestellt.

**Genereller Hinweis zum Geodatensatz:**


Weitere Informationen zu diesem Datensatz finden sie unter [«**Züri wie neu»** auf Geocat.ch](https://www.geocat.ch/geonetwork/srv/ger/catalog.search#/metadata/1561c795-fcb7-4a59-889d-cd9d6d8fecf2).				
			   

**Informationen zum Datensatz:**

Die Meldungen auf Züri wie Neu sind zusätzlich auch noch über eine Open311-API abrufbar. Siehe dazu 
<a style='font-weight:bold' href='https://www.zueriwieneu.ch/open311/' target='_blank'>https://www.zueriwieneu.ch/open311/</a>


**Datenerfassung:**

Online Datenerfassung durch die Bevölkerung auf https://www.stadt-zuerich.ch/zueriwieneu

**Statisches Vorschaubild:**

![BildText](https://www.gis.stadt-zuerich.ch/zueriplan_docs/geocat/1561c795-fcb7-4a59-889d-cd9d6d8fecf2.png)



## Data set links

[Direct link by OpenDataZurich for dataset](https://data.stadt-zuerich.ch/dataset/geo_zueri_wie_neu)

https://www.stadt-zuerich.ch/geodaten/download/Zueri_wie_neu?format=geojson_link<br>


## Metadata
- **Publisher** `GIS-Zentrum, Geomatik + Vermessung, Tiefbau- und Entsorgungsdepartement`
- **Maintainer** `Open Data Zürich`
- **Maintainer_email** `opendata@zuerich.ch`
- **Keywords** `Bauen und Wohnen,Bevölk­erung,Energie,Umwelt,Ver­walt­ung`
- **Tags** `['geodaten', 'geoportal', 'infrastruktur', 'mangelmelder', 'punktdaten', 'schaden', 'stzh', 'vektordaten']`
- **Metadata_created** `2023-11-06T03:11:40.167288`
- **Metadata_modified** `2026-04-18T06:33:52.024520`


## Imports and helper functions

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import json
import xml.etree.ElementTree as ET
import requests
import geopandas as gpd
import re
import io
import contextily as ctx
import matplotlib.patches as mpatches
import numpy as np
from mapclassify import NaturalBreaks

In [2]:
# helper function for reading datasets with proper separator
def get_dataset(url):
    # get URL ofs WFS service
    dataset_identifier = re.search(r'\/([^\/\?]+)\?', url).group(1)
    url_geoportal = f"https://www.ogd.stadt-zuerich.ch/wfs/geoportal/{dataset_identifier}"
    print("Getting available layers from:", url_geoportal)
    
    # Parameter for GetCapabilities
    params = {
        "service": "WFS",
        "version": "1.1.0",
        "request": "GetCapabilities"
    }
    
    # send GetCapabilities
    response = requests.get(url_geoportal, params=params)

    # parse XML answer
    root = ET.fromstring(response.content)

    # define Namespace 
    namespace = {'wfs': 'http://www.opengis.net/wfs'}
    
    # Exctract available layers
    layers = [feature_type.find('wfs:Name', namespace).text for feature_type in root.findall('.//wfs:FeatureType', namespace)]
    
    print("Available layers:", layers)
    print("First layer is set as default. To chose another layer set it as typename in the get_dataset() function.")

    # set first layer als typename
    typename = layers[0]
    print("Chosen typename:", typename)

    # Parameter GetFeature request
    params = {
        "service": "WFS",
        "version": "1.1.0",
        "request": "GetFeature",
        "typename": typename,
        "outputFormat": "application/json"
    }
    
    # GetFeature request
    response = requests.get(url_geoportal, params=params)

    # Load GeoJSON in GeoDataFrame
    gdf = gpd.read_file(io.StringIO(json.dumps(response.json())))
    return gdf

## Load the data

In [3]:
gdf = get_dataset('https://www.stadt-zuerich.ch/geodaten/download/Zueri_wie_neu?format=geojson_link')

Getting available layers from: https://www.ogd.stadt-zuerich.ch/wfs/geoportal/Zueri_wie_neu
Available layers: ['zwn_meldungen_p']
First layer is set as default. To chose another layer set it as typename in the get_dataset() function.
Chosen typename: zwn_meldungen_p


## Analyze the data

gdf.plot()

In [4]:
# drop columns that have no values
gdf.dropna(how='all', axis=1, inplace=True)

In [5]:
print(f'The dataset has {gdf.shape[0]:,.0f} rows (observations) and {gdf.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {gdf.duplicated().sum()} exact duplicates in the data.')

The dataset has 73,312 rows (observations) and 20 columns (variables).
There seem to be 0 exact duplicates in the data.


In [6]:
gdf.info(memory_usage='deep', verbose=True)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 73312 entries, 0 to 73311
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    73312 non-null  str           
 1   agency_sent_datetime  72479 non-null  datetime64[ms]
 2   description           73312 non-null  str           
 3   detail                73312 non-null  str           
 4   e                     73312 non-null  int32         
 5   interface_used        73312 non-null  str           
 6   media_url             73312 non-null  str           
 7   n                     73312 non-null  int32         
 8   objectid              73312 non-null  int32         
 9   requested_datetime    73312 non-null  datetime64[ms]
 10  service_code          73312 non-null  str           
 11  service_name          73312 non-null  str           
 12  service_notice        73312 non-null  str           
 13  service_

In [7]:
gdf.head()

,id,agency_sent_datetime,description,detail,e,interface_used,media_url,n,objectid,requested_datetime,service_code,service_name,service_notice,service_request_id,status,title,updated_datetime,url,userid,geometry
0,zwn_meldungen_p.1,2013-04-04 07:25:05,Auf dem Asp: Auf dem Asphalt des Bürgersteigs ...,Auf dem Asphalt des Bürgersteigs hat es eine E...,2678968,Web interface,,1247548,1,2013-03-14 15:16:15,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,Diese Reparatur wird von uns in den kommenden ...,1,fixed - council,Auf dem Asp,2013-04-12 07:59:30,https://www.zueriwieneu.ch/report/1,16624,POINT (8.48423 47.37404)
1,zwn_meldungen_p.2,2013-03-26 14:05:05,Vermessungs: Vermessungspunkt ist nicht mehr b...,Vermessungspunkt ist nicht mehr bündig mit dem...,2680746,Web interface,,1249916,2,2013-03-14 15:17:57,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,Diese Reparatur wird von uns in den kommenden ...,2,fixed - council,Vermessungs,2013-04-12 08:00:22,https://www.zueriwieneu.ch/report/2,16624,POINT (8.50819 47.39512)
2,zwn_meldungen_p.3,2013-03-15 09:55:05,Beim Trotto: Beim Trottoir sind einige Randste...,Beim Trottoir sind einige Randsteine defekt un...,2684605,Web interface,https://www.zueriwieneu.ch/photo/4.0.jpeg?bfbb...,1251431,3,2013-03-15 09:14:16,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,Diese Reparatur wird von uns in den kommenden ...,4,fixed - council,Beim Trotto,2013-04-12 08:08:10,https://www.zueriwieneu.ch/report/4,16624,POINT (8.55959 47.40826)
3,zwn_meldungen_p.4,2013-03-20 10:05:05,Auf dem Par: Auf dem Parkplatz beim Waidspital...,Auf dem Parkplatz beim Waidspital sind einige ...,2681754,Web interface,https://www.zueriwieneu.ch/photo/5.0.jpeg?e309...,1250376,4,2013-03-15 09:17:15,Strasse/Trottoir/Platz,Strasse/Trottoir/Platz,Diese Reparatur wird von uns in den kommenden ...,5,fixed - council,Auf dem Par,2013-04-12 08:09:05,https://www.zueriwieneu.ch/report/5,16624,POINT (8.52163 47.39913)
4,zwn_meldungen_p.5,2013-04-22 18:25:05,Arbeitskist: Arbeitskiste ist rund herum versc...,Arbeitskiste ist rund herum verschmiert,2683094,Web interface,https://www.zueriwieneu.ch/photo/6.0.jpeg?8e65...,1247762,5,2013-03-15 10:36:53,Abfall/Sammelstelle,Abfall/Sammelstelle,Dieses Graffiti wird von uns in den kommenden ...,6,fixed - council,Arbeitskist,2013-04-23 13:50:33,https://www.zueriwieneu.ch/report/6,16624,POINT (8.53889 47.37546)


In [8]:
# display a small random sample transposed in order to see all variables
gdf.sample(3).T

,50617,34223,68731
id,zwn_meldungen_p.50581,zwn_meldungen_p.34186,zwn_meldungen_p.68711
agency_sent_datetime,2024-05-23 06:27:04,2022-06-02 13:32:02,2025-12-08 07:27:05
description,Busch wächs: Busch wächst bei Parkbank,Ein Brett b: Ein Brett bzw. Schaltafel,Sperrgut La: Sperrgut Lampe auf Trottoir
detail,Busch wächst bei Parkbank,Ein Brett bzw. Schaltafel,Sperrgut Lampe auf Trottoir
e,2683936,2681607,2680694
interface_used,desktop,desktop,iOS
media_url,https://www.zueriwieneu.ch/photo/56587.0.jpeg?...,https://www.zueriwieneu.ch/photo/37600.0.jpeg?...,https://www.zueriwieneu.ch/photo/76662.0.jpeg?...
n,1251720,1247953,1247076
objectid,50581,34186,68711
requested_datetime,2024-05-22 11:21:06,2022-06-02 13:29:08,2025-12-08 07:22:58


**Contact**: opendata@zuerich.ch

Now repeat the Steps for gdf_kreise

In [16]:
# read Gpk, make sure it has the correct CRS (LV95) 
gdf_kreise = gpd.read_file(r'C:\Users\volt4\OneDrive\Desktop\_Studium\8_Semster\SdS\sds210\Project\Data\Geometry_Stadtkreise\data\Stadtkreise_geometry.gpkg', layer='stzh.adm_stadtkreise_v')
print("Current CRS:", gdf_kreise.crs)




Current CRS: EPSG:2056


In [ ]:
# drop columns that have no values
gdf_kreise.dropna(how='all', axis=1, inplace=True)

In [ ]:
print(f'The dataset has {gdf_kreise.shape[0]:,.0f} rows (observations) and {gdf_kreise.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {gdf_kreise.duplicated().sum()} exact duplicates in the data.')

In [ ]:
gdf_kreise.info(memory_usage='deep', verbose=True)

In [ ]:
gdf_kreise.head()

In [ ]:
# display a small random sample transposed in order to see all variables
gdf_kreise.sample(3).T